# Chapter 14: What Geometric Algebra Gave Us

## A retrospective on the journey from vectors to holonomy

We began this tutorial series in Chapter 1 with a deceptively simple observation: *vectors live somewhere*. A hidden-state vector in a transformer is not just a list of numbers — it is an element of a high-dimensional vector space that the model has learned to structure. The question was: how do we see that structure?

Over thirteen chapters, we built a progressively richer geometric vocabulary for answering that question. We started with the dot product and the outer product, moved to bivectors and rotors, learned to read principal planes, decompose layer transitions into grade-0 and grade-2 parts, measure curvature through holonomy, detect non-commutativity through commutator bivectors, and finally applied all of these tools to practical diagnostics.

The central thesis of this series has been:

> **Matrix algebra tells you *how much*. Geometric algebra tells you *how much* and *in which direction*.**

Every scalar summary we computed in the pre-GA framework — $\|U - I\|_F$, $\|[A_i, A_j]\|_F$, $\|\Omega - I\|_F$ — has a GA counterpart that carries strictly more information. The rotor gives a magnitude (rotation angle) *and* a direction (the bivector generator). The holonomy gives a curvature magnitude *and* a curvature plane. The commutator gives a non-commutativity strength *and* the plane of non-commutativity.

This final chapter summarizes the entire journey, catalogs the complete matrix-to-GA bridge, identifies open problems, and points you toward further reading.

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga

print("Imports OK")
print(f"NumPy {np.__version__}")

## The Journey: Chapter by Chapter

The following table maps each chapter in this tutorial series to its core GA concept and the transformer insight it enables:

| Chapter | Title | GA Concept | Transformer Insight |
|---------|-------|-----------|-------------------|
| 1 | Vectors Live Somewhere | Vector spaces, inner products | Hidden states are geometric objects, not just arrays |
| 2 | The Product That Does Everything | Geometric product $ab = a \cdot b + a \wedge b$ | Every pair of hidden states has both a scalar similarity and a plane of interaction |
| 3 | Planes, Not Axes | Bivectors $e_i \wedge e_j$ | Rotations are specified by planes, not axes; this works in any dimension |
| 4 | When Coordinates Lie | Whitening, metric estimation | The model's internal metric differs from the Euclidean metric; we must whiten before measuring geometry |
| 5 | Rotations Without Matrices | Rotors $R = \exp(-B\theta/2)$ | Layer transitions decompose into a rotor (rotation) and a metric deformation (stretch) |
| 6 | What a Layer Does | Versor decomposition $T = R \cdot P$ | Grade-0 (stretch) and grade-2 (rotation) tell complementary stories about each layer |
| 7 | Reading the Planes | Principal plane decomposition | The top few planes carry most of the rotation; plane tracking reveals layer specialization |
| 8 | The Eigenvalue Story | Spectrum of $P$, condition number $\kappa$ | High $\kappa$ means the layer is selective; effective rank measures dimensionality |
| 9 | When Order Matters | Commutator $[B_i, B_j]$ | Non-commuting rotations create new geometry; this is compositional capacity |
| 10 | Curvature from Holonomy | Holonomy rotor $R_{\text{loop}}$ | Curvature on the layer-time grid measures path dependence of the representation |
| 11 | Dependency and Effective Work | $D_\ell$, weighted capacity $C_{\text{eff}}$ | Not all geometry is functional; dependency separates signal from noise |
| 12 | Scaling and Capacity | $C_{\text{acc}}$, $C_{\text{eff}}$, concentration | Larger models have more non-commutative capacity and higher concentration in final layers |
| 13 | Diagnosing and Steering | Context detection, hallucination, steering | GA gives plane-specific diagnostics and rank-2 interventions |
| **14** | **What GA Gave Us** | **Retrospective** | **The complete matrix-to-GA bridge** |

Each row builds on the previous ones. The progression from scalars (Ch. 1-4) to bivectors (Ch. 5-7) to fields (Ch. 8-10) to applications (Ch. 11-14) mirrors the standard progression in differential geometry: from points, to tangent vectors, to connections, to curvature.

## What Was Invisible Before

Before Geometric Algebra, the layer-time geometric framework operated with scalar summaries. These summaries are *correct* — they measure real geometric quantities — but they are *incomplete*. Here is precisely what GA made visible:

### 1. Rotation Planes

**Before (matrix):** $\|U^{(\ell)} - I\|_F$ tells us how much layer $\ell$ rotates. A value of 0.5 means "moderate rotation."

**After (GA):** The bivector $B^{(\ell)} = \log(R^{(\ell)})$ tells us the rotation angle *and* the rotation plane. Two layers can have the same $\|U - I\|_F$ but rotate in completely different planes — meaning they transform completely different aspects of the representation.

### 2. Curvature Planes

**Before (matrix):** $\|\Omega^{(\ell)} - I\|_F$ tells us how much the layer-time grid curves at layer $\ell$. High curvature means "path-dependent transport."

**After (GA):** The holonomy bivector $B_{\text{loop}} = \log(R_{\text{loop}})$ tells us the curvature magnitude *and* the curvature plane. If the curvature plane shifts when we add context to the prompt, the model is routing information through different subspaces — a much stronger signal than a magnitude change.

### 3. Commutator Planes

**Before (matrix):** $\|[A_i, A_j]\|_F$ tells us how much layers $i$ and $j$ fail to commute. Non-zero commutator means "order matters."

**After (GA):** The commutator bivector $[B_i, B_j]$ tells us the magnitude of non-commutativity *and* the plane in which it occurs. If the commutator lives in the same plane as the dominant rotation, the non-commutativity is *aligned* with the computation. If it lives in an orthogonal plane, the non-commutativity creates *new* geometric directions.

### 4. Functionally Relevant Planes

By combining bivector decomposition with dependency weighting, we can identify not just *any* plane of rotation, but the **functionally relevant** planes — those that contribute to the final output. This enables plane-specific steering (Chapter 13, Application 3), which is impossible with scalar summaries alone.

The common thread: **GA replaces scalars with bivectors**, and bivectors carry directional information that scalars discard.

## The Matrix-Multivector Bridge

The following table provides the complete mapping between matrix operations (as used in the original layer-time geometry framework) and their GA equivalents (as implemented in `layer_time_ga`):

| Matrix Operation | GA Equivalent | Object Type | What It Measures |
|-----------------|---------------|-------------|-----------------|
| Orthogonal matrix $U$ | Rotor $R = \exp(-B/2)$ | `Rotor` | Rotation part of layer transition |
| Skew-symmetric $A = \log(U)$ | Bivector generator $B$ | `Bivector` | Plane and angle of rotation |
| Symmetric PD matrix $P$ | Metric deformation (grade-0) | `np.ndarray` | Stretch spectrum of layer transition |
| Polar decomposition $T = UP$ | Versor decomposition $T = R \cdot P$ | `VersorDecomposition` | Full grade decomposition |
| $\|U - I\|_F$ | Rotor angle $\theta = \|B\|_F / \sqrt{2}$ | `float` | Rotation magnitude |
| SVD of $A$ | Principal plane decomposition | `list[dict]` | Dominant rotation planes and weights |
| $[A_i, A_j] = A_i A_j - A_j A_i$ | $[B_i, B_j]$ commutator bivector | `Bivector` | Non-commutativity plane and magnitude |
| $\|[A_i, A_j]\|_F$ | Commutator norm $\|[B_i, B_j]\|_F$ | `float` | Non-commutativity strength |
| Loop product $\Omega = P_1 P_2^{-1}$ | Holonomy rotor $R_{\text{loop}}$ | `HolonomyResult` | Curvature rotor (plane + magnitude) |
| $\|\Omega - I\|_F$ | Holonomy scalar curvature | `float` | Curvature magnitude |
| $\log(\Omega)$ | Holonomy bivector $B_{\text{loop}}$ | `Bivector` | Curvature plane |
| Condition number $\kappa(P)$ | Same (grade-0 quantity) | `float` | Metric selectivity |
| Effective rank of $P$ | Same (grade-0 quantity) | `float` | Metric dimensionality |
| $(M + M^T)/2$ | Grade-0 projection | `np.ndarray` | Symmetric (stretch) part |
| $(M - M^T)/2$ | Grade-2 projection | `Bivector` | Antisymmetric (rotation) part |

Notice that grade-0 quantities (condition number, effective rank, singular values) are the same in both frameworks — they describe the *metric deformation*, which is a scalar-valued quantity. The power of GA appears in grade-2 quantities: wherever the matrix framework produces a scalar norm, GA produces a bivector with both magnitude and direction.

In [ ]:
# --- Load model and run a comprehensive analysis ---
# This demonstrates the full pipeline in one place: from text to complete GA profile.

model = ltg_ga.load_model("Qwen/Qwen2.5-7B")

prompt = "Geometric algebra reveals the hidden structure of transformers by"
result = ltg_ga.analyse(prompt, model=model, compute_dependency=True)

print(f"Model: {model.name}")
print(f"Prompt: \"{prompt}\"")
print(f"Layers: {result.n_layers}  |  Tokens: {result.n_tokens}  |  Whitened dim: {result.k}")
print(f"Bivectors: {len(result.bivector_field)} layer transitions")
print(f"Holonomy map shape: {result.holonomy_map.shape}")
print()
print("Full pipeline complete: hidden states -> whitening -> rotor field ->")
print("holonomy curvature -> commutator structure -> dependency -> all diagnostics.")

In [ ]:
# --- Full GA summary: all quantities in one printout ---
from layer_time_ga.curvature import commutator_field, commutator_plane_decomposition

print("=" * 65)
print("  COMPLETE GA ANALYSIS SUMMARY")
print("=" * 65)
print()

# --- Rotor field statistics ---
print("-- Rotor Field --")
print(f"  Number of layer transitions: {len(result.bivector_field)}")
print(f"  Mean rotation angle:         {result.angles.mean():.4f} rad")
print(f"  Max rotation angle:          {result.angles.max():.4f} rad (layer {result.angles.argmax()})")
print(f"  Min rotation angle:          {result.angles.min():.4f} rad (layer {result.angles.argmin()})")
print(f"  Std of angles:               {result.angles.std():.4f} rad")
print()

# Top-3 rotating layers
top3 = np.argsort(result.angles)[-3:][::-1]
print(f"  Top-3 rotating layers: {top3.tolist()}")
for l in top3:
    biv = result.bivector_field[l]
    planes = biv.principal_planes(n_planes=1)
    specificity = "N/A"
    if planes:
        p2 = biv.principal_planes(n_planes=2)
        if len(p2) >= 2:
            specificity = f"{p2[0]['weight'] / (p2[0]['weight'] + p2[1]['weight']):.3f}"
    print(f"    Layer {l}: angle={result.angles[l]:.4f}, "
          f"bivector norm={biv.norm:.4f}, plane specificity={specificity}")
print()

# --- Holonomy (curvature) statistics ---
print("-- Holonomy Curvature --")
curv = result.curvature_by_layer
if curv.size > 0:
    print(f"  Mean curvature:              {curv.mean():.4f}")
    print(f"  Peak curvature:              {curv.max():.4f} (layer {curv.argmax()})")
    print(f"  Total curvature:             {curv.sum():.4f}")
    total_curv = curv.sum()
    if total_curv > 0:
        final3 = curv[-3:].sum() / total_curv
        first3 = curv[:3].sum() / total_curv
        print(f"  First-3-layer share:         {first3:.1%}")
        print(f"  Final-3-layer share:         {final3:.1%}")
else:
    print(f"  (no plaquettes available)")
print()

# --- Commutator structure ---
print("-- Commutator Structure --")
comm_norms = commutator_field(result.bivector_field)
upper_tri = comm_norms[np.triu_indices_from(comm_norms, k=1)]
print(f"  Total commutator mass:       {upper_tri.sum():.4f}")
print(f"  Mean pairwise ||[B_i,B_j]||: {upper_tri.mean():.4f}")
print(f"  Max pairwise ||[B_i,B_j]||:  {upper_tri.max():.4f}")

# Find the most non-commuting pair
max_idx = np.unravel_index(np.argmax(comm_norms), comm_norms.shape)
print(f"  Most non-commuting pair:     layers ({max_idx[0]}, {max_idx[1]})")

# Principal commutator planes
plane_info = commutator_plane_decomposition(result.bivector_field, n_planes=3)
print(f"  Total commutator norm:       {plane_info['total_norm']:.4f}")
if plane_info['principal_planes']:
    print(f"  Top principal planes:")
    for i, p in enumerate(plane_info['principal_planes']):
        print(f"    Plane {i+1}: weight = {p['weight']:.4f}")
print()

# --- Metric deformation ---
print("-- Metric Deformation (Grade-0) --")
print(f"  Mean condition number:       {result.condition_numbers.mean():.2f}")
print(f"  Max condition number:        {result.condition_numbers.max():.2f} "
      f"(layer {result.condition_numbers.argmax()})")
print(f"  Mean effective rank:         {result.effective_ranks.mean():.1f}")
print(f"  Min effective rank:          {result.effective_ranks.min():.1f} "
      f"(layer {result.effective_ranks.argmin()})")
print()

# --- Dependency profile ---
if result.dependency_profile is not None:
    print("-- Dependency Profile --")
    print(f"  Total dependency:            {result.dep_total:.4f}")
    print(f"  Dependency entropy:          {result.dep_entropy:.3f}")
    print(f"  Horizon-90:                  layer {result.dep_horizon_90}")
    dep = result.dependency_profile
    peak_dep = int(np.argmax(dep))
    print(f"  Peak dependency layer:       {peak_dep} (D = {dep[peak_dep]:.6f})")
print()
print("=" * 65)

## Open Problems

The GA lens on transformer geometry is new, and there are many directions for future work. Here are five open problems that we find particularly interesting:

### 1. Higher Grades

Our framework uses grade-0 (scalars, metric deformation) and grade-2 (bivectors, rotation planes). But the Clifford algebra $\operatorname{Cl}(k, 0)$ has elements of every grade from 0 to $k$. **Grade-4 elements** (trivectors, quadrivectors) could capture higher-order geometric structure. For instance, a grade-4 element $e_i \wedge e_j \wedge e_k \wedge e_l$ represents a 4-dimensional subspace. Do transformer layers ever perform transformations that are best described by these higher-grade elements? The polar decomposition only extracts grades 0 and 2, but a *full* multivector decomposition might reveal richer structure.

### 2. Conformal Geometric Algebra (CGA)

Standard GA works in flat Euclidean space. **Conformal GA** embeds $\mathbb{R}^k$ into $\mathbb{R}^{k+1,1}$ (adding one extra dimension of each signature), which enables translations, dilations, and inversions to be represented as rotors. In our framework, translations are currently invisible — we decompose $T = R \cdot P$ but discard any translational component. CGA would let us track *both* rotation and translation of hidden states through the layer stack, potentially revealing how the model moves representations to different "regions" of activation space.

### 3. GA in Training Dynamics

All of our analysis has been *post hoc*: we load a trained model and examine its geometry. But the same tools could be applied to **training trajectories**. How do rotor angles, curvature profiles, and commutator structure evolve during training? Does capacity emerge gradually or in sudden phase transitions? Tracking bivector plane alignment across training checkpoints could reveal when and how the model learns to specialize different layers.

### 4. Multimodal Models

Vision-language models process multiple modalities through shared transformer layers. The GA framework naturally extends to this setting: we can compare the rotation planes used for visual tokens versus text tokens. If the planes are orthogonal, the model is processing modalities in independent subspaces. If the planes overlap, the model is integrating modalities. **Cross-modal bivector alignment** could be a powerful diagnostic for understanding multimodal fusion.

### 5. Projective Geometric Algebra (PGA)

**Projective GA** uses the algebra $\mathbb{R}^*_{k,0,1}$ (with one degenerate dimension) to represent points, lines, and planes as algebraic objects. In this setting, a hidden-state vector is a *point* in projective space, and a layer transition maps points to points. The projective framework naturally handles the fact that hidden states can have different norms (which in standard GA we factor out via the polar decomposition). PGA might provide a more unified treatment of the metric deformation and rotation.

## Your Next Steps

### Further Reading

**On Geometric Algebra itself:**
- Doran & Lasenby, *Geometric Algebra for Physicists* (Cambridge, 2003) — the definitive reference for GA with applications in physics. Chapters 1-5 cover the algebra; Chapters 6-8 cover rotors and spinors.
- Dorst, Fontijne & Mann, *Geometric Algebra for Computer Science* (Morgan Kaufmann, 2007) — a more applied treatment with algorithms and implementations.
- Hestenes, *New Foundations for Classical Mechanics* (Kluwer, 2nd ed., 1999) — shows how GA simplifies classical mechanics; the rotor formalism for rotations is particularly elegant.

**On transformer interpretability:**
- Elhage et al., "A Mathematical Framework for Transformer Circuits" (Anthropic, 2021) — the mechanistic interpretability perspective on how transformer layers compose.
- Geva et al., "Transformer Feed-Forward Layers Build Predictions by Promoting Concepts" (EMNLP, 2022) — how FFN layers modify representations.

**On the layer-time geometry framework:**
- Sudjianto & Zhang, *Layer-Time Geometry of Large Language Models* (2026) — the monograph on which this tutorial series is based.

### The `layer_time_ga` Code Library

The code library that powers these tutorials is organized as follows:

| Module | Key Functions | Purpose |
|--------|--------------|---------|
| `ltg_ga` | `load_model()`, `analyse()`, `compare()`, `capacity()` | High-level student API |
| `layer_time_ga.algebra` | `Bivector`, `Rotor`, `bivector_from_skew()`, `rotor_from_orthogonal()`, `commutator_bivector()`, `grade_decomposition()` | GA primitives |
| `layer_time_ga.decomposition` | `versor_decompose()`, `extract_rotor_field()`, `extract_bivector_field()`, `extract_metric_field()` | Layer decomposition |
| `layer_time_ga.curvature` | `holonomy_rotor()`, `holonomy_field()`, `holonomy_scalar_map()`, `commutator_field()`, `commutator_plane_decomposition()` | Curvature and commutators |
| `layer_time_ga.capacity` | `ga_capacity_profile()` | Compositional capacity |
| `layer_time_ga.plotting` | `plot_rotor_angle_profile()`, `plot_holonomy_map()`, `plot_commutator_heatmap()`, `plot_ga_summary()` | Visualization |

### Extending the Analysis

To apply the GA framework to your own research:

1. **New models:** Simply call `ltg_ga.load_model("your/model")` with any Hugging Face causal LM.
2. **New diagnostics:** Use `result.bivector_field` to access raw bivectors, then compute any custom similarity or alignment metric you need.
3. **Batch analysis:** Loop over a dataset of prompts, calling `ltg_ga.analyse()` on each, and aggregate the GA quantities (e.g., mean plane specificity, mean commutator concentration) to characterize model behavior at scale.
4. **Cross-model comparison:** Run the same prompts through different models and compare their GA profiles using `ltg_ga.compare()`.

In [ ]:
# --- Capstone: Generate the complete GA summary plot ---
# This is the single figure that captures the full geometric story.

save_path = os.path.join(os.getcwd(), 'ch14_capstone_summary.png')
result.plot_ga_summary(save_path=save_path)

print(f"Capstone GA summary plot saved to: {save_path}")
print()
print("This four-panel figure is the canonical output of the GA framework:")
print("  (1) Rotor angles — rotation magnitude at each layer")
print("  (2) Holonomy curvature — where the geometry is curved")
print("  (3) Metric deformation — condition number and effective rank")
print("  (4) Commutator heatmap — which layer pairs interact non-trivially")
print()
print("Every panel tells a part of the story. Together, they give a complete")
print("geometric portrait of how the transformer processes this prompt.")

## Congratulations!

You have completed the full tutorial series on **Geometric Algebra for Transformer Interpretability**.

Over fourteen chapters, you learned to:

- Represent hidden-state transformations as **rotors** and **bivectors** instead of raw matrices
- Decompose layer transitions into **grade-0** (metric deformation) and **grade-2** (rotation) components
- Extract **principal rotation planes** and track how they evolve across the layer stack
- Measure **curvature** through holonomy rotors on the layer-time grid
- Quantify **compositional capacity** through bivector non-commutativity
- Separate geometric activity from functional relevance using **dependency weighting**
- Apply the full toolkit to **practical diagnostics**: context-ignoring detection, hallucination signatures, and plane-specific steering

The key takeaway is that Geometric Algebra does not replace the matrix framework — it *enriches* it. Every matrix computation we performed before still holds; GA simply reveals the directional structure that scalar summaries discard. Wherever the matrix framework gives a number, GA gives a number *and* a plane.

The `layer_time_ga` library and the `ltg_ga` student API make these tools accessible for undergraduate research projects. We encourage you to apply them to your own models, your own prompts, and your own questions about how transformers work.

*Happy exploring.*